In [9]:
# imports
import pandas as pd
import matplotlib as plt
import math

# display options
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [10]:
years = [
    2011,
    2012,
    2013,
    2014,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026,
]

In [11]:
def getRowAmount(row):
    if not pd.isna(row["_ordinance_amount_"]):
        return row["_ordinance_amount_"]
    elif not pd.isna(row["appropriation_ordinance"]):
        return row["appropriation_ordinance"]
    else:
        return row["amount"]


def fmtCode(code, length):
    return "0" * (length - len(str(code))) + str(code)

In [12]:
df = pd.DataFrame()
for year in years:
    dfYear = pd.read_csv(f"../data/{year}-ordinance.csv")
    dfYear["year"] = year
    df = pd.concat([df, dfYear], ignore_index=True)

functionalCategories = pd.read_csv("../labels/categories.csv", index_col=0)[
    "category"
]

# clean rows and columns
df["amount"] = df.apply(getRowAmount, axis=1)
df["fund_code"] = df.apply(lambda row: fmtCode(row["fund_code"], 4), axis=1)
df["department_number"] = df.apply(
    lambda row: fmtCode(row["department_number"], 2), axis=1
)
df["appropriation_account"] = df.apply(
    lambda row: fmtCode(row["appropriation_account"], 4), axis=1
)
df["appropriation_authority"] = df["appropriation_authority"].apply(
    lambda x: str(int(x)) if isinstance(x, float) and not math.isnan(x) else x
)
df["appropriation_authority"] = df.apply(
    lambda row: fmtCode(row["appropriation_authority"], 4), axis=1
)
df["functional_category"] = df.apply(
    lambda row: functionalCategories.loc[int(row["department_number"])], axis=1
)

df = df.drop(
    columns=["department", "_ordinance_amount_", "appropriation_ordinance"]
)

df.head()

,fund_type,fund_code,fund_description,department_number,department_description,appropriation_authority,appropriation_authority_description,appropriation_account,appropriation_account_description,amount,year,functional_category
0,LOCAL,0100,CORPORATE FUND,55,POLICE BOARD,2005,2005 - POLICE BOARD,0050,STIPENDS,145000.00,2011,Public Safety
1,LOCAL,0100,CORPORATE FUND,55,POLICE BOARD,2005,2005 - POLICE BOARD,0130,POSTAGE,300.00,2011,Public Safety
2,LOCAL,0100,CORPORATE FUND,39,BOARD OF ELECTION COMMISSIONER,2005,2005 - ELECTION AND ADMIN DIVISION,0005,SALARIES AND WAGES - ON PAYROLL,6831849.00,2011,Legislative & Elections
3,LOCAL,0100,CORPORATE FUND,39,BOARD OF ELECTION COMMISSIONER,2005,2005 - ELECTION AND ADMIN DIVISION,0015,SCHEDULE SALARY ADJ,27342.00,2011,Legislative & Elections
4,LOCAL,0100,CORPORATE FUND,39,BOARD OF ELECTION COMMISSIONER,2005,2005 - ELECTION AND ADMIN DIVISION,0030,LESS SALARY SAVINGS FROM UNPAID TIME OFF,-607288.00,2011,Legislative & Elections


In [15]:
df[df["department_number"] == "72"]["department_description"].value_counts()

department_description
DEPARTMENT OF ENVIRONMENT                     88
Department of Environment                     83
Office of Climate and Environmental Equity     1
Name: count, dtype: int64